In [1]:
#imports

import numpy as np

In [2]:
#(audio) file input

In [ ]:
# simple autocorrelation function (prone to "too high" mis-detections ??)

def acor(x, tau, W, t0=0):
    '''
    inputs:
        x[t]    - time-domain discrete audio signal (assuming 1D-np-array), 
        tau     - lag / candidate period, measured in samples
        t0      - starting sample of window W
        W       - window of autocorrelation function

    returns:
        autocorrelation value r_t0(tau) = Σ_{j=0}^{W-1} x[t+j] · x[t+j+tau]
        r_t0(0) will always yield the global maximum (autocorrelation is maximized when comparing to itself)
    '''

    #r_t0 = 0
    #for j in range(W):
    #    r_0 += x[t0+j]*x[t0+j+tau]

    # need to convert to float64 to resolve any wraparound issue with 16-bit-WAV-audio (int16) files
    # the largest int16 product is ((-2)^15)*((-2)^15) = 2^30, with one bit being the sign bit
    # any value above that gets wrapped around
    # the largest integer float32 can represent EXACT is 2^24, while for float64 it is 2^53
    x = np.asarray(x, dtype=np.float64)

    if x.size < t0+W+tau:
        raise IndexError("SizeError: End of signal reached")
    r_t0 = np.sum(x[t0:t0+W]*x[t0+tau:t0+W+tau])

    return r_t0

In [ ]:
# YIN algorithm

# Difference function
def diff(x, tau, W, t0=0):
    '''
    inputs:
        x[t]    - time-domain discrete audio signal (assuming 1D-np-array), 
        tau     - lag / candidate period, measured in samples
        t0      - starting sample of window W
        W       - window of autocorrelation function

    returns:
        difference function according to d_t(τ) = Σ_{j=0}^{W-1} ( x[t+j] − x[t+j+τ] )²
    '''

    if x.size < t0+W+tau:
            raise IndexError("SizeError: End of signal reached")

    x = np.asarray(x, dtype=np.float64)

    d_t = 0
    for j in range(W):
        d_t += (x[t0+j] - x[t0+j+tau])**2

        # for C++ implementation
        '''
        xa = x[t0+j]
        xb = x[t0+j+tau]
        d_t += (xa*xa - 2*xa*xb + xb*xb)
        '''

        # -> fft-based implementation?

    return d_t


# Cumulative mean normalized difference (CMND)
def diff_cmnd(x, maxtau, W, t0=0):
    '''
    inputs:
        x[t]    - time-domain (sample-wise) discrete audio signal (assuming 1D-np-array), 
        tau     - lag / candidate period, measured in samples
        t0      - starting sample of window W
        W       - window of autocorrelation function

    returns:
        difference prime function according to  - d'(0) = 1
                                                - d'(τ) = d(τ) / [ (1/τ) · Σ_{j=1}^{τ} d(j) ]
        tau_global_min, which is the global minimum 
    '''

    # maybe outsource the computation of d(j) over all samples up to tau to another functions,
    # which returns the values of all possible tau values to an np array. Will do if many such computations

    if(maxtau==0):
         return 1.0

    d_t = np.ones(maxtau)
    for j in range(maxtau):
         d_t[j+1] = diff(x, j+1, W, t0)

    d_prime = d_t[maxtau]*maxtau/np.sum(d_t)

    return d_prime


# absolute threshold
def abs_threshold(x, maxtau, W, t0=0, threshold=0.1):
    '''
    inputs:
        x[t]        - time-domain (sample-wise) discrete audio signal (assuming 1D-np-array), 
        tau         - lag / candidate period, measured in samples
        t0          - starting sample of window W
        W           - window of autocorrelation function
        threshold   - absolute threshold value for tau selection

    outputs:
        tau_min     - first tau to reach sub-threshold-value, or global argmin tau

    '''

    d_prime = diff_cmnd(x, maxtau, W, t0)

    for i in range(maxtau):
         if d_prime[i] <= threshold:
              return i

    # if no tau reaches the threshold, choose global minimum
    tau_global_min = np.argmin(d_prime)
    return tau_global_min


# parabolic interpolation
def par_interpol(x, maxtau, W, t0=0, threshold=0.1):
    '''
    inputs:
        x[t]        - time-domain (sample-wise) discrete audio signal (assuming 1D-np-array), 
        tau         - lag / candidate period, measured in samples
        t0          - starting sample of window W
        W           - window of autocorrelation function
        threshold   - absolute threshold value for tau selection
    outputs:
        tau_star    - the parabolic approximation of the tau chosen in abs_threshold()
    '''

    chosen_tau = abs_threshold(x, maxtau, W, t0, threshold)
    d_chosen_tau_neighbors = diff(x, maxtau=0, W=3, t0=chosen_tau-1)

    # τ* = τ + (d[τ−1] − d[τ+1]) / (2·(d[τ−1] − 2·d[τ] + d[τ+1]))
    tau_star = chosen_tau + (d_chosen_tau_neighbors[0]-d_chosen_tau_neighbors[2]) / (2*(d_chosen_tau_neighbors[0]-2*d_chosen_tau_neighbors[1]+d_chosen_tau_neighbors[2]))

    return tau_star

In [ ]:
# test cases for acor(x, tau, W, t0)
# each test prints what it expected and what it got; a crash in one test does not stop the others

# The tests come out as expected, two of them fail because the window W is too small s.t. r(0) is not globally maximized 

rng = np.random.default_rng(0)
results = []

def check(name, expected, got, tol=1e-9):
    '''compare numbers/arrays (relative tolerance) or anything else (==), print both, record PASS/FAIL'''
    try:
        e = np.asarray(expected, dtype=float)
        g = np.asarray(got, dtype=float)
        ok = e.shape == g.shape and np.allclose(g, e, rtol=tol, atol=tol)
    except (TypeError, ValueError):
        ok = expected == got
    results.append((name, ok))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}")
    print(f"        expected: {expected}")
    print(f"        got:      {got}")

def run(name, test):
    '''run one test; an unexpected exception counts as FAIL and is printed'''
    try:
        test()
    except Exception as err:
        results.append((name, False))
        print(f"[FAIL] {name}")
        print(f"        raised {type(err).__name__}: {err}")

def ref_acor(x, tau, W, t0=0):
    '''slow reference: the loop from the docstring, in float64'''
    return sum(float(x[t0+j]) * float(x[t0+j+tau]) for j in range(W))

def peak_lag(x, W, tau_min, tau_max, t0=0):
    '''lag in [tau_min, tau_max] where acor is largest'''
    r = np.array([acor(x, tau, W, t0) for tau in range(tau_min, tau_max + 1)])
    return tau_min + int(np.argmax(r))

fs = 44100  # sample rate [Hz]


# 1) zero lag equals the energy of the window
def t_zero_lag():
    x = rng.standard_normal(1000)
    for W, t0 in [(1, 0), (50, 0), (256, 100), (500, 499)]:
        check(f"1  zero lag = window energy (W={W}, t0={t0})",
              np.sum(x[t0:t0+W]**2), acor(x, 0, W, t0))
run("1  zero lag", t_zero_lag)


# 2) matches the slow reference loop from the docstring, for many (tau, W, t0)
def t_reference_loop():
    x = rng.standard_normal(300)
    cases = [(tau, W, t0) for tau in (0, 1, 7, 50) for W in (1, 10, 64) for t0 in (0, 3, 100)]
    exp = [ref_acor(x, *c) for c in cases]
    got = [acor(x, *c) for c in cases]
    check(f"2  matches docstring loop on {len(cases)} (tau, W, t0) combos", np.round(exp, 6), np.round(got, 6))
run("2  reference loop", t_reference_loop)


# 3) comparison against np.correlate
#    np.correlate(a, v, 'valid')[k] = sum_n a[n+k] * v[n], so with a = x[t0 : t0+W+max_tau] and v = x[t0 : t0+W]
#    entry k is r_t0(k)
def t_np_correlate():
    x = rng.standard_normal(2000)
    W, t0, max_tau = 400, 123, 300
    exp = np.correlate(x[t0:t0+W+max_tau], x[t0:t0+W], mode='valid')
    got = np.array([acor(x, tau, W, t0) for tau in range(max_tau + 1)])
    check(f"3  equals np.correlate for tau = 0..{max_tau} (showing max |difference|)",
          0.0, np.max(np.abs(exp - got)))
run("3  np.correlate", t_np_correlate)


# 4) pure sines: searching lags between half a period and 1.5 periods, the peak sits at the period
#    (the true period is usually not a whole number of samples, so the integer peak lag should be within 1 sample of it)
def t_sines():
    W = 2048
    for f in [100.0, 116.54, 233.08, 441.0, 882.0]:   # 116.54 Hz = Bb2, the trombone's open-position fundamental
        period = fs / f
        t = np.arange(W + round(1.5 * period) + 1) / fs
        x = np.sin(2*np.pi*f*t)
        got = peak_lag(x, W, max(1, round(0.5 * period)), round(1.5 * period))
        check(f"4  sine {f} Hz: peak lag within 1 sample of fs/f = {period:.2f}  (showing True/False, lag below)",
              True, abs(got - period) <= 1)
        print(f"        peak lag = {got}")
run("4  sines", t_sines)


# 5) sine at an exact integer period: r(P) should equal r(0), and r(P/2) should equal -r(0)
def t_sine_integer_period():
    P, W = 100, 1000          # W is a whole number of periods
    x = np.sin(2*np.pi*np.arange(3000)/P)
    r0 = acor(x, 0, W)
    check("5  integer period P=100: r(P) = r(0)", r0, acor(x, P, W))
    check("5  integer period P=100: r(P/2) = -r(0)", -r0, acor(x, P//2, W))
    check("5  integer period P=100: r(P/4) = 0 (sin vs cos)", 0.0, round(acor(x, P//4, W), 9))
run("5  integer period", t_sine_integer_period)


# 6) harmonic-rich, trombone-like tone: fundamental Bb2 plus 8 harmonics, 2nd and 3rd louder than the fundamental
def t_trombone_like():
    f0, W = 116.54, 2048
    tau_min, tau_max = fs // 1000, fs // 50
    amps = [0.6, 1.0, 0.9, 0.7, 0.5, 0.35, 0.25, 0.15, 0.1]
    t = np.arange(W + tau_max + 1) / fs
    x = sum(a * np.sin(2*np.pi*(k+1)*f0*t + 0.3*k) for k, a in enumerate(amps))
    period = fs / f0
    check(f"6  trombone-like Bb2: largest peak in 50-1000 Hz range at round(fs/f0) = {round(period)}",
          round(period), peak_lag(x, W, tau_min, tau_max), tol=1 / period)
    # with some noise added, searching between half a period and 1.5 periods
    xn = x + 0.3 * rng.standard_normal(x.size)
    got = peak_lag(xn, W, round(0.5 * period), round(1.5 * period))
    check(f"6  trombone-like Bb2 + noise: peak lag within 1 sample of fs/f0 = {period:.2f}  (showing True/False, lag below)",
          True, abs(got - period) <= 1)
    print(f"        peak lag = {got}")
run("6  trombone-like", t_trombone_like)


# 7) impulses
def t_impulse():
    x = np.zeros(100); x[10] = 1.0
    got = [acor(x, tau, W=20) for tau in range(0, 30)]
    exp = [1.0] + [0.0] * 29
    check("7  single impulse at n=10, W=20: r(0)=1, r(tau)=0 otherwise", exp, got)
    x = np.zeros(100); x[10] = 1.0; x[35] = 1.0     # two impulses 25 samples apart
    got = [acor(x, tau, W=20) for tau in range(0, 30)]
    exp = [1.0] + [0.0] * 24 + [1.0] + [0.0] * 4
    check("7  two impulses 25 apart, W=20: r(0)=1, r(25)=1, else 0", exp, got)
run("7  impulse", t_impulse)


# 8) silence
def t_silence():
    x = np.zeros(1000)
    got = [acor(x, tau, W=256) for tau in (0, 1, 100, 500)]
    check("8  silence: all zeros, no NaN", [0.0, 0.0, 0.0, 0.0], got)
run("8  silence", t_silence)


# 9) scaling: r is quadratic in the amplitude, r(a*x) = a^2 * r(x)
def t_scaling():
    x = rng.standard_normal(500)
    for a in (2.0, -3.0, 0.01):
        check(f"9  scaling by a={a}: r(a*x) = a^2 * r(x)", a**2 * acor(x, 37, 100, 5), acor(a * x, 37, 100, 5))
run("9  scaling", t_scaling)


# EDITED: CHANGED W FROM 300
# 10) normalization / Cauchy-Schwarz: |r_t0(tau)| <= sqrt(E(x[t0:t0+W]) * E(x[t0+tau:t0+tau+W]))
#     i.e. r / sqrt(E1*E2) always lies in [-1, 1], and equals 1 when the two windows are identical
def t_normalization():
    x = rng.standard_normal(2000)
    W, t0 = 250, 50
    worst = 0.0
    for tau in range(0, 1000):
        E1 = np.sum(x[t0:t0+W]**2); E2 = np.sum(x[t0+tau:t0+tau+W]**2)
        worst = max(worst, abs(acor(x, tau, W, t0)) / np.sqrt(E1 * E2))
    check("10 normalized |r|/sqrt(E1*E2) <= 1 for all tau (showing True/False, max value below)", True, worst <= 1 + 1e-12)
    print(f"        max normalized |r| = {worst:.6f}")
    P = 150
    xp = np.tile(rng.standard_normal(P), 20)          # exactly periodic, period 150, 20 times
    E1 = np.sum(xp[t0:t0+W]**2); E2 = np.sum(xp[t0+P:t0+P+W]**2)
    check("10 periodic noise (P=150): normalized r(P) = 1", 1.0, acor(xp, P, W, t0) / np.sqrt(E1 * E2))
run("10 normalization", t_normalization)


# 11) docstring claim: r_t0(0) is the global maximum over tau
#     checked on a few different signals; prints the largest r(tau)/r(0) over tau = 1..400 for each
def t_docstring_max():
    W, n = 256, 256 + 401
    t = np.arange(n)
    signals = {
        "white noise":            rng.standard_normal(n),
        "steady sine (P=64)":     np.sin(2*np.pi*t/64),
        "crescendo sine (P=64)":  np.sin(2*np.pi*t/64) * np.linspace(0.1, 1.0, n),
        "decaying sine (P=64)":   np.sin(2*np.pi*t/64) * np.linspace(1.0, 0.1, n),
        "note starting mid-window": np.where(t >= 200, np.sin(2*np.pi*t/64), 0.0),
    }
    for name, x in signals.items():
        r0 = acor(x, 0, W)
        rmax = max(acor(x, tau, W) for tau in range(1, 401))
        check(f"11 docstring claim r(0) >= r(tau) for all tau: {name}", True, bool(r0 >= rmax))
        print(f"        r(0) = {r0:.4f}, max over tau>0 = {rmax:.4f}")
run("11 docstring max", t_docstring_max)


# 12) window offset: acor(x, tau, W, t0) = acor(x[t0:], tau, W, 0)
def t_offset():
    x = rng.standard_normal(600)
    for t0 in (0, 1, 250):
        check(f"12 offset t0={t0} equals the same window on x[t0:]", acor(x[t0:], 40, 128, 0), acor(x, 40, 128, t0))
run("12 offset", t_offset)


# 13) time reversal: r of the reversed signal, with the window placed at the mirrored position, is the same number
#     (the two windows x[t0:t0+W] and x[t0+tau:t0+tau+W] just swap roles)
def t_time_reversal():
    x = rng.standard_normal(500); N = x.size
    for tau, W, t0 in [(0, 50, 10), (13, 64, 0), (100, 200, 150)]:
        t0_rev = N - (t0 + W + tau)
        check(f"13 time reversal symmetry (tau={tau}, W={W}, t0={t0})", acor(x, tau, W, t0), acor(x[::-1], tau, W, t0_rev))
run("13 time reversal", t_time_reversal)


# 14) lag sign symmetry: sum_j x[t0+j]*x[t0+j-tau] is the same sum as r at start t0-tau with lag +tau
#     so acor(x, -tau, W, t0) is expected to equal acor(x, tau, W, t0-tau)  (or to be rejected with an error)
def t_negative_lag():
    x = rng.standard_normal(200)
    tau, W, t0 = 5, 20, 50
    exp = acor(x, tau, W, t0 - tau)
    try:
        got = acor(x, -tau, W, t0)
    except Exception as err:
        got = f"{type(err).__name__}: {err}"
    check(f"14 negative lag: acor(x, -{tau}, W, t0) = acor(x, {tau}, W, t0-{tau}) or an error", exp, got)
run("14 negative lag", t_negative_lag)


# 15) end of signal: the last lag whose window still fits (t0+W+tau == len(x)) works,
#     one lag further raises IndexError
def t_boundary():
    x = np.array([1, 2, 3, 4, 5, 6, -7, 8, 9, 10], dtype=float)     # the example from the cell above
    W, t0 = 3, 0
    last = x.size - W - t0                                           # = 7
    check(f"15 last fitting lag tau={last} on the 10-sample example", ref_acor(x, last, W, t0), acor(x, last, W, t0))
    try:
        acor(x, last + 1, W, t0); got = "no error"
    except IndexError:
        got = "IndexError"
    except Exception as err:
        got = type(err).__name__
    check(f"15 lag tau={last + 1} runs past the end", "IndexError", got)
    check("15 full table tau=0..7 on the example", [ref_acor(x, tau, W) for tau in range(last + 1)],
          [acor(x, tau, W) for tau in range(last + 1)])
run("15 boundary", t_boundary)


# 16) input types: plain int array (like the example above) and int16 samples as read from a 16-bit WAV file
def t_dtypes():
    xi = np.array([1, 2, 3, 4, 5, 6, -7, 8, 9, 10])
    check("16 int array gives the same as float", acor(xi.astype(float), 2, 3), acor(xi, 2, 3))
    x16 = (np.sin(2*np.pi*np.arange(1000)/50) * 30000).astype(np.int16)   # loud 16-bit audio
    check("16 int16 WAV-style samples (W=500, tau=50)", ref_acor(x16, 50, 500), acor(x16, 50, 500), tol=1e-6)
run("16 dtypes", t_dtypes)


# summary
n_pass = sum(ok for _, ok in results)
print(f"\n{n_pass} / {len(results)} checks passed")

[PASS] 1  zero lag = window energy (W=1, t0=0)
        expected: 0.01580808849619356
        got:      0.01580808849619356
[PASS] 1  zero lag = window energy (W=50, t0=0)
        expected: 42.340036474218635
        got:      42.340036474218635
[PASS] 1  zero lag = window energy (W=256, t0=100)
        expected: 271.14477711070464
        got:      271.14477711070464
[PASS] 1  zero lag = window energy (W=500, t0=499)
        expected: 442.3772576523336
        got:      442.3772576523336
[PASS] 2  matches docstring loop on 36 (tau, W, t0) combos
        expected: [ 1.4016240e+00  7.0726000e-02  2.3505000e-02  4.8713200e+00
  3.3490050e+00  7.2875520e+00  6.6033545e+01  7.0474053e+01
  4.5553291e+01  3.5970100e-01 -3.6332100e-01  7.4783000e-02
  7.4175100e-01  2.9936600e-01  6.0610150e+00 -7.0195630e+00
 -9.2714700e+00  1.1622080e+00  2.3371800e-01 -2.0555000e-02
  8.3559000e-02  8.3273100e-01  6.7453800e-01 -1.4569100e-01
  5.7537340e+00  5.8779120e+00 -2.1141400e+00  2.2611600e-01
 -2